# 01 — Data Inventory & Exploration
**Day 1, Step 2.** Goal: understand every raw dataset before writing any cleaning or modeling code — what each table contains, its size, its time range, its join keys, and its obvious quality issues.

This notebook does **not** modify any data. It only reads and reports. Cleaning happens in `02_data_cleaning_and_validation.ipynb` (Day 2).

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.utils.config_loader import load_config, raw_path

cfg = load_config()


## 1. Dataset overview
The technical report describes 12 linked datasets (Site, Meteorological, Long-Term Reference,
Turbine Master, Power Curve, SCADA, Alarms, Maintenance, Grid, GIS/Terrain, Weather Forecast,
Layout). What actually shipped in `Datasets.zip` is close to that, at demo scale. We load a small
sample of each first — several of these files are large (the 20-year climate file is >1M rows,
~110 MB) so don't `.read_csv()` the whole thing without thinking.

In [2]:
# Quick shape + column check for every raw file, without loading huge files fully.
summary_rows = []
for key, filename in cfg["raw_files"].items():
    path = raw_path(key, cfg)
    # count lines cheaply instead of loading the whole file into memory
    with open(path, "rb") as f:
        n_lines = sum(1 for _ in f) - 1  # minus header
    sample = pd.read_csv(path, nrows=5, encoding="latin1")
    summary_rows.append({
        "key": key,
        "filename": filename,
        "rows": n_lines,
        "columns": sample.shape[1],
        "size_mb": round(path.stat().st_size / 1e6, 1)
    })

summary_df = pd.DataFrame(summary_rows).sort_values("size_mb", ascending=False)
summary_df


,key,filename,rows,columns,size_mb
0,long_term_climate,Dataset-4 Long term wind climate dataset 20 ye...,1051200,12,111.2
3,alarms,Dataset-7 Turbine alarm event dataset 3year 50...,150000,20,39.7
2,scada,Dataset-6 Wind farm scada 1year.csv,52560,32,12.0
4,maintenance,Dataset-8 Wind turbine maintenance dataset 3y...,25000,17,4.9
10,weather_forecast,Dataset-12 weather forecast 10000 rows.csv,10000,20,1.5
5,grid_curtailment,Dataset-9 Grid and Curtailment dataset 10000 r...,10000,17,1.3
8,layout_candidates,Dataset-11 (B) AI Candidate Layouts 10000.csv,10000,21,1.3
9,layout_position_features,Dataset-11 (C) Layout Position Features 10000.csv,10000,14,1.0
11,simple_flat_dataset,wind_energy_dataset_100- 10000 Row.csv,10000,10,0.7
1,power_curve,Dataset-5 Turbine power curve dataset.csv,305,6,0.0


## 2. SCADA dataset — the core forecasting table (Dataset-6)
This is the one Day 3's forecasting model will be built on.

In [3]:
scada = pd.read_csv(raw_path("scada", cfg), parse_dates=["timestamp"])
print("Shape:", scada.shape)
print("Date range:", scada["timestamp"].min(), "to", scada["timestamp"].max())
print("Turbines:", scada["turbine_id"].nunique())
scada.head()


Shape: (52560, 32)
Date range: 2025-01-01 00:00:00 to 2025-12-31 23:50:00
Turbines: 1


,timestamp,wind_farm_id,turbine_id,wind_speed_mps,wind_direction_deg,wind_gust_mps,active_power_kw,reactive_power_kvar,energy_generated_kwh,rotor_speed_rpm,generator_speed_rpm,pitch_angle_1_deg,pitch_angle_2_deg,pitch_angle_3_deg,yaw_position_deg,yaw_error_deg,gearbox_temperature_c,generator_temperature_c,bearing_temperature_c,nacelle_temperature_c,hydraulic_pressure_bar,oil_temperature_c,oil_pressure_bar,ambient_temperature_c,air_pressure_hpa,air_density_kg_m3,grid_voltage_v,grid_frequency_hz,availability_status,operating_status,curtailment_status,alarm_code
0,2025-01-01 00:00:00,WF_IND_001,WTG_001,8.05,241.2,9.51,699.30,56.04,116.55,10.79,964.47,1.90,2.51,1.86,244.1,-3.76,69.36,57.98,49.18,30.19,141.52,57.69,3.36,18.84,1009.81,1.2048,695.09,50.010,Available,Generating,Not Curtailed,NONE
1,2025-01-01 00:10:00,WF_IND_001,WTG_001,5.71,208.9,7.39,233.11,39.68,38.85,9.54,869.42,2.56,2.40,2.13,210.5,-3.22,61.32,44.75,40.00,25.57,148.42,52.01,3.16,19.91,1013.76,1.2051,693.52,49.989,Available,Generating,Not Curtailed,NONE
2,2025-01-01 00:20:00,WF_IND_001,WTG_001,9.01,228.0,10.85,1013.01,43.42,168.83,11.53,1034.66,1.25,2.26,1.96,227.6,5.41,73.21,60.54,50.28,29.67,147.65,59.09,3.41,20.29,1022.48,1.2139,695.49,49.949,Available,Generating,Not Curtailed,NONE
3,2025-01-01 00:30:00,WF_IND_001,WTG_001,9.43,245.9,10.74,1274.91,32.53,212.49,11.90,1093.48,1.85,1.80,2.45,245.8,-1.83,76.76,72.01,52.54,31.17,140.82,59.66,2.98,19.23,1022.51,1.2183,688.88,50.012,Available,Generating,Not Curtailed,NONE
4,2025-01-01 00:40:00,WF_IND_001,WTG_001,4.30,252.5,6.46,77.36,80.63,12.89,8.34,761.73,2.45,1.58,1.98,253.7,4.96,55.14,46.30,38.41,26.60,148.98,49.41,3.20,18.24,1001.15,1.1969,690.23,49.982,Available,Generating,Not Curtailed,NONE


In [4]:
scada.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52560 entries, 0 to 52559
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   timestamp                52560 non-null  datetime64[ns]
 1   wind_farm_id             52560 non-null  object        
 2   turbine_id               52560 non-null  object        
 3   wind_speed_mps           52560 non-null  float64       
 4   wind_direction_deg       52560 non-null  float64       
 5   wind_gust_mps            52560 non-null  float64       
 6   active_power_kw          52560 non-null  float64       
 7   reactive_power_kvar      52560 non-null  float64       
 8   energy_generated_kwh     52560 non-null  float64       
 9   rotor_speed_rpm          52560 non-null  float64       
 10  generator_speed_rpm      52560 non-null  float64       
 11  pitch_angle_1_deg        52560 non-null  float64       
 12  pitch_angle_2_deg        52560 n

In [5]:
# Quick data-quality pass, using the validation rules from the technical report (Section 17)
issues = {
    "negative_wind_speed": (scada["wind_speed_mps"] < 0).sum(),
    "power_above_zero_with_zero_rotor_speed": ((scada["rotor_speed_rpm"] == 0) & (scada["active_power_kw"] > 0)).sum(),
    "wind_direction_out_of_range": (~scada["wind_direction_deg"].between(0, 360)).sum(),
    "missing_active_power": scada["active_power_kw"].isna().sum(),
    "duplicate_timestamps_per_turbine": scada.duplicated(subset=["timestamp", "turbine_id"]).sum(),
}
pd.Series(issues, name="count")


negative_wind_speed                       0
power_above_zero_with_zero_rotor_speed    0
wind_direction_out_of_range               0
missing_active_power                      0
duplicate_timestamps_per_turbine          0
Name: count, dtype: int64

## 3. Turbine power curve (Dataset-5)
Used to sanity-check whether SCADA's actual power roughly follows the expected curve, and later to compute Expected Power − Actual Power = Energy Loss (Report §16).

In [6]:
power_curve = pd.read_csv(raw_path("power_curve", cfg))
print("Turbine models:", power_curve["turbine_model"].unique())
power_curve.head()


Turbine models: ['WTG_1500_77' 'WTG_2000_100' 'WTG_2500_110' 'WTG_3000_120' 'WTG_5000_150']


,turbine_model,wind_speed_mps,expected_power_kw,thrust_coefficient,rotor_speed_rpm,pitch_angle_deg
0,WTG_1500_77,0.0,0.0,0.0,0.0,0.0
1,WTG_1500_77,0.5,0.0,0.0,0.0,0.0
2,WTG_1500_77,1.0,0.0,0.0,0.0,0.0
3,WTG_1500_77,1.5,0.0,0.0,0.0,0.0
4,WTG_1500_77,2.0,0.0,0.0,0.0,0.0


## 4. Alarms / events (Dataset-7) and maintenance (Dataset-8)
These two are the labels for anomaly detection and predictive maintenance (Report Models 3 & 4).

In [7]:
alarms = pd.read_csv(raw_path("alarms", cfg), parse_dates=["event_timestamp", "event_end_timestamp"])
print("Alarms shape:", alarms.shape)
print("Date range:", alarms["event_timestamp"].min(), "to", alarms["event_timestamp"].max())
alarms["alarm_category"].value_counts()


Alarms shape: (150000, 20)
Date range: 2023-01-01 00:06:00 to 2025-12-31 17:59:00


alarm_category
Temperature      45968
Electrical       25668
Control          17019
Mechanical       15076
Operational      14179
Communication    12904
Hydraulic         9340
Lubrication       7661
Safety            2185
Name: count, dtype: int64

In [8]:
maintenance = pd.read_csv(raw_path("maintenance", cfg), parse_dates=["failure_date"])
print("Maintenance shape:", maintenance.shape)
maintenance["component"].value_counts()


Maintenance shape: (25000, 17)


component
Brake System           2571
Generator              2550
Hydraulic System       2540
Blade/Pitch            2514
Main Bearing           2500
Yaw System             2491
Gearbox                2485
Power Converter        2485
SCADA/Communication    2473
Transformer            2391
Name: count, dtype: int64

## 5. Grid & curtailment (Dataset-9)
Grid Agent input — where potential energy is lost to curtailment (Report §12 industry-style loss structure).

In [9]:
grid = pd.read_csv(raw_path("grid_curtailment", cfg), parse_dates=["timestamp"])
print("Grid shape:", grid.shape)
grid[["potential_energy_kwh", "actual_exported_energy_kwh", "energy_not_delivered_kwh"]].describe()


Grid shape: (10000, 17)


,potential_energy_kwh,actual_exported_energy_kwh,energy_not_delivered_kwh
count,10000.000000,10000.000000,10000.000000
mean,237.701993,219.600017,18.101976
std,159.584676,156.218924,52.495741
min,0.000000,0.000000,0.000000
25%,101.257500,83.557500,1.510000
50%,215.605000,195.875000,4.740000
75%,370.147500,344.605000,10.932500
max,500.000000,499.980000,491.550000


## 6. Layout datasets (Dataset-11 A/B/C)
A = the real demo farm's turbine positions. B/C = 10,000 candidate layouts with computed AEP and wake loss — this is training data for a future layout-optimization model, not something to build on Day 1, but worth knowing it exists.

In [10]:
layout_actual = pd.read_csv(raw_path("layout_actual", cfg))
layout_candidates = pd.read_csv(raw_path("layout_candidates", cfg))
print("Actual farm turbines:", layout_actual.shape[0])
print("Candidate layouts:", layout_candidates.shape[0])
layout_actual.head()


Actual farm turbines: 100
Candidate layouts: 10000


,farm_id,turbine_id,latitude,longitude,elevation_m,hub_height_m,rotor_diameter_m,turbine_model,distance_to_nearest_turbine_m,dominant_wind_sector_deg,land_constraint,terrain_slope_deg,terrain_roughness,access_road_distance_m,grid_connection_distance_m
0,WF001,T001,11.233709,77.453143,261.2,140,170,WTG-6.0MW,813.9,180,Road,6.39,0.090,501.9,4364.2
1,WF001,T002,11.285564,77.513641,293.1,140,130,WTG-4.2MW,591.9,0,Road,11.68,0.187,784.0,11116.7
2,WF001,T003,11.265879,77.481436,295.3,140,170,WTG-6.0MW,464.9,135,Slope,8.63,0.376,809.3,4135.6
3,WF001,T004,11.253879,77.500857,260.0,120,130,WTG-5.0MW,617.8,315,NaN,10.52,0.309,1023.4,9679.9
4,WF001,T005,11.214042,77.540757,350.3,130,170,WTG-5.0MW,203.0,90,Slope,13.26,0.250,1095.6,7667.0


## 7. Weather forecast (Dataset-12) and long-term climate (Dataset-4)
Dataset-4 is large (~1M rows, 20 years). Load only a slice for exploration — never load it fully inside a notebook you'll re-run often.

In [11]:
weather_forecast = pd.read_csv(raw_path("weather_forecast", cfg), parse_dates=["forecast_issue_time", "forecast_valid_time"])
print("Weather forecast shape:", weather_forecast.shape)
weather_forecast[["wind_speed_forecast_error_mps", "power_forecast_error_kw"]].describe()


Weather forecast shape: (10000, 20)


,wind_speed_forecast_error_mps,power_forecast_error_kw
count,10000.000000,10000.000000
mean,-0.005107,-57.479388
std,0.656985,261.322982
min,-3.438000,-1948.830000
25%,-0.366000,-135.260000
50%,-0.001000,-15.020000
75%,0.357000,49.302500
max,3.226000,1814.220000


In [12]:
# Only read a slice of the 20-year file for exploration
climate_sample = pd.read_csv(raw_path("long_term_climate", cfg), nrows=200_000, parse_dates=["timestamp"])
print("Sample shape:", climate_sample.shape)
print("Sites:", climate_sample["site_id"].unique())
climate_sample["wind_speed_hub_height_mps"].describe()


Sample shape: (200000, 12)
Sites: ['SITE_001']


count    200000.000000
mean          7.750075
std           1.771994
min           2.285000
25%           6.499000
50%           7.637000
75%           8.879000
max          16.903000
Name: wind_speed_hub_height_mps, dtype: float64

## 8. GIS/Terrain (Dataset-10) and the simple flat dataset
GIS/Terrain is static, not time series. The `wind_energy_dataset_100-10000 Row.csv` file is a
simpler, flat table — good for a quick smoke test but not the primary modeling dataset (no
turbine-level detail, no maintenance/alarm linkage).

In [13]:
gis = pd.read_csv(raw_path("gis_terrain", cfg))
print("GIS rows:", gis.shape)
gis.head()


GIS rows: (100, 12)


,latitude,longitude,elevation_m,slope_deg,aspect_deg,terrain_roughness,land_cover,surface_roughness_length,obstacle_distance_m,obstacle_height_m,road_distance_m,substation_distance_m
0,11.0,78.00000,200,0.0,0,0.10,Grassland,0.03,50,5,100,500
1,11.0,78.00027,205,1.3,17,0.18,Shrubland,0.10,62,8,120,550
2,11.0,78.00054,210,2.6,34,0.26,Agriculture,0.17,74,11,140,600
3,11.0,78.00081,215,3.9,51,0.34,Barren,0.24,86,14,160,650
4,11.0,78.00108,220,5.2,68,0.42,Grassland,0.31,98,17,180,700


In [14]:
simple = pd.read_csv(raw_path("simple_flat_dataset", cfg), encoding="latin1")
print("Simple dataset shape:", simple.shape)
simple.head()


Simple dataset shape: (10000, 10)


,Date_Time,Turbine_ID,Wind Speed (m/s),Wind Gust (m/s),Wind Direction (°),Temperature (°C),Humidity percent (%),Air Pressure (hPa),Rotor Speed (RPM),Power Generated (kW)
0,01-01-2025 00:00,WT-004,4.61,10.27,134.5,22.59,64.91,1017.56,10.25,0.00
1,01-01-2025 01:00,WT-004,10.73,13.22,119.8,26.76,65.06,1014.23,24.25,1206.86
2,01-01-2025 02:00,WT-003,7.37,8.13,63.4,28.93,57.35,1017.39,15.70,248.34
3,01-01-2025 03:00,WT-001,6.24,6.49,218.6,21.34,79.07,1013.52,12.18,128.44
4,01-01-2025 04:00,WT-004,2.90,4.88,171.6,30.16,61.18,1007.60,5.83,19.89


## 9. Join-key check
The technical report's whole point (Section 13/16) is that these tables are *relational*, joined
on `site_id` / `turbine_id` / `timestamp` — not one flat table. Confirm the keys actually line up
before Day 2's merge.

In [15]:
print("SCADA turbine_ids:      ", sorted(scada['turbine_id'].unique())[:5], "...")
print("Alarms turbine_ids:      ", sorted(alarms['turbine_id'].unique())[:5], "...")
print("Maintenance turbine_ids: ", sorted(maintenance['turbine_id'].unique())[:5], "...")
print("Grid turbine_ids:        ", sorted(grid['turbine_id'].unique())[:5], "...")

common_turbines = (set(scada['turbine_id']) & set(alarms['turbine_id'])
                    & set(maintenance['turbine_id']) & set(grid['turbine_id']))
print("\nTurbines common to all four tables:", len(common_turbines))


SCADA turbine_ids:       ['WTG_001'] ...
Alarms turbine_ids:       ['WTG_001', 'WTG_002', 'WTG_003', 'WTG_004', 'WTG_005'] ...
Maintenance turbine_ids:  ['WTG_001', 'WTG_002', 'WTG_003', 'WTG_004', 'WTG_005'] ...
Grid turbine_ids:         ['WTG_001', 'WTG_002', 'WTG_003', 'WTG_004', 'WTG_005'] ...

Turbines common to all four tables: 1


---
## Day 1 checkpoint

You should now be able to answer, for every dataset:
- What are its columns and rough size?
- What is its time range?
- Does it share join keys (`turbine_id`, `site_id`, `timestamp`) with the others?
- What obvious quality issues does it have?

**Write down (in `reports/summaries/day1_data_notes.md`) anything that surprised you** — e.g. how
many turbines are actually common across tables, any date-range mismatches between SCADA (1 year)
and maintenance/alarms (3 years), and the missing-demand-dataset gap flagged in the README.

Next: `02_data_cleaning_and_validation.ipynb` (Day 2) — apply the validation rules from Report
Section 17, resolve the issues found above, and produce the first `data/interim/` table.